# Gradio Chatbot


In [30]:

from openai import OpenAI

import gradio as gr

In [31]:
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [32]:
MODEL_GEMMA = "gemma2:2b"
MODEL_LLAMA = "llama3.2"

---

## Simple Chatbot

**In the chatbot below, notice that history is autofed into the callback function**

In [33]:
def chat(message, history):
    return f"You said {message} and the history is {history} but I still say bananas"

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


**Parse the autofed history in the callback function**

In [34]:
system_message = "You are a helpful assistant"

def chat(message,history):
    # parse history into message format without metadata field
    history = [{"role": item["role"], "content": item["content"]} for item in history]

    # componse the message to the LLM
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    # get LLM response
    response = client.chat.completions.create(model=MODEL_GEMMA, messages=messages)

    return response.choices[0].message.content



In [35]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


---

## Streaming output in chatbot

In [36]:
def stream_chat(message,history):
    # parse history into message format without metadata field
    history = [{"role": item["role"], "content": item["content"]} for item in history]

    # componse the message to the LLM
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    # get LLM response
    stream = client.chat.completions.create(model=MODEL_LLAMA, messages=messages, stream=True)

    response = ""

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response    

In [37]:
gr.ChatInterface(fn=stream_chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


---

## One-shot prompting

In [38]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

Ask: "hi - what do you guys have"

In [39]:
gr.ChatInterface(fn=stream_chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


In [40]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

Ask: "do you sell shoes?"

In [41]:
gr.ChatInterface(fn=stream_chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


---

## Augmenting system prompt conditionally

In [ ]:
def conditional_augmentation_chat(message, history):

    # extract and parse history
    history = [{"role": item["role"], "content": item["content"]} for item in history]

    # save raw system message
    raw_system_message = system_message

    # conditonal augementation of system message
    if 'belt' in message.lower():
        raw_system_message += "The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."

    # setup the chat with the LLM
    messages = [{"role": "system", "content": raw_system_message}] + history + [{"role": "user", "content": message}]
    stream = client.chat.completions.create(model=MODEL_LLAMA, messages=messages, stream=True)

    # extract stream output
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response


Ask: 'do you have belts?"

In [46]:
gr.ChatInterface(fn=conditional_augmentation_chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/luz-vndra/Documents/GitHub/ai-notebooks/llm-engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/luz-vndra/Documents/GitHub/ai-notebooks/llm-engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/luz-vndra/Documents/GitHub/ai-notebooks/llm-engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/luz-vndra/Documents/GitHub/ai-notebooks/llm-engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1710, in call_function
    prediction = await utils.async_iteration(iterator)
                 